# Microglia Ilastik (Active Learning) Localization

This notebook contains the code for the execution and evaluation of microglia localization using a model trained with [Ilastik](https://www.ilastik.org/) pixel classification.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.utils import *
from ilastik import app
from ilastik.applets.dataSelection.opDataSelection import PreloadedArrayDatasetInfo
import vigra
from scipy import ndimage as ndi
from skimage.segmentation import watershed
import pyvips
from tqdm import tqdm
import os


def start_ilastik(project_path):
    """Parse args to run headless ilastik prediction for specified project"""

    args = app.parse_args([])
    args.headless = True
    args.project = project_path
    shell = app.main(args)
    return shell


def predict_ilastik(shell, tile_np):
    """Given a tile and args for ilastik prediction, run prediction"""

    tile_tagged = vigra.taggedView(tile_np, "yxc")

    data = [
        {
            "Raw Data": PreloadedArrayDatasetInfo(
                preloaded_array=tile_tagged,
                axistags=vigra.defaultAxistags("yxc"),
            )
        }
    ]

    preds = shell.workflow.batchProcessingApplet.run_export(
        data,
        export_to_array=True,
    )
    return preds[0]


os.environ["LAZYFLOW_THREADS"] = "2"
os.environ["LAZYFLOW_TOTAL_RAM_MB"] = "3000"

path = "./ilastik_files/segmentation_final.ilp"

data_dir = "./evaluation/train"
batch_size = 20
files = [f for f in Path(data_dir).glob("./*.png")]
shell = start_ilastik(path)
preds_dict = {}
preds_dict_pts = {}
stride = 50

for file in files:
    img = pyvips.Image.new_from_file(file, access="sequential")
    for processed, img_np, (y_patch, x_patch) in tqdm(
        create_patches(img, 512, stride=512 / 2, filter=filter_tissue),
        desc="process patches",
    ):

        pred = predict_ilastik(shell, img_np)
        # centroids
        labels = np.argmax(pred, axis=-1)
        mask = (labels == 0).astype("uint8")
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

        cc, n = ndi.label(mask)
        centroids = ndi.center_of_mass(mask, cc, range(1, n + 1))
        centroids = np.array([(int(round(y)), int(round(x))) for y, x in centroids])
        # watershed
        masked = (labels < 2).astype("uint8")
        kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        masked = cv2.morphologyEx(masked, cv2.MORPH_CLOSE, kernel2)

        dist = ndi.distance_transform_edt(masked)
        markers = np.zeros(masked.shape, dtype=np.int32)
        for i, (y, x) in enumerate(centroids, start=1):
            markers[int(y), int(x)] = i
        labels = np.array(watershed(-dist, markers, mask=masked))
        labels, segmented_cells = filter_size(labels, len(centroids))
        # bounding boxes
        bounding_boxes = calculate_bb(segmented_cells)
        final_bb = []
        for bb in bounding_boxes:
            x1, y1, x2, y2 = bb
            if (x2 - x1) * (y2 - y1) < 16000 or (x2 - x1) < 25 or (y2 - y1) < 25:
                continue
            else:
                final_bb.append(bb)
        for x, y, x2, y2 in final_bb:
            cell = img.crop(x, y, (x2 - x), (y2 - y))

## Evaluation

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
from scripts.filters import *
from scripts.utils import *
from scripts.evaluate import *
from ilastik import app
from ilastik.applets.dataSelection.opDataSelection import PreloadedArrayDatasetInfo
import vigra
from scipy import ndimage as ndi
from skimage.segmentation import watershed
import pyvips
from tqdm import tqdm
import os


def start_ilastik(project_path):
    """Parse args to run headless ilastik prediction for specified project"""
    args = app.parse_args([])
    args.headless = True
    args.project = project_path
    shell = app.main(args)
    return shell


def predict_ilastik(shell, tile_np):
    """Given a tile and args for ilastik prediction, run prediction"""

    tile_tagged = vigra.taggedView(tile_np, "yxc")

    data = [
        {
            "Raw Data": PreloadedArrayDatasetInfo(
                preloaded_array=tile_tagged,
                axistags=vigra.defaultAxistags("yxc"),
            )
        }
    ]

    preds = shell.workflow.batchProcessingApplet.run_export(
        data,
        export_to_array=True,
    )
    return preds[0]


os.environ["LAZYFLOW_THREADS"] = "2"
os.environ["LAZYFLOW_TOTAL_RAM_MB"] = "3000"

path = "./ilastik_files/segmentation_final.ilp"
ground_truth = read_annotations("./evaluation/annotations/gt_bb.json")
ground_truth_pts = read_annotations_points("./evaluation/annotations/gt_points.json")

data_dir = "./evaluation/test"
batch_size = 20
files = [f for f in Path(data_dir).glob("./*.png")]
shell = start_ilastik(path)
preds_dict = {}
preds_dict_pts = {}

for idx, file in tqdm(enumerate(files)):

    img = pyvips.Image.new_from_file(file, access="sequential")

    img_np = np.ndarray(
        buffer=img.write_to_memory(),
        dtype=np.uint8,
        shape=[img.height, img.width, img.bands],
    )

    pred = predict_ilastik(shell, img_np)
    # centroids
    labels = np.argmax(pred, axis=-1)
    mask = (labels == 0).astype("uint8")
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)

    cc, n = ndi.label(mask)
    centroids = ndi.center_of_mass(mask, cc, range(1, n + 1))
    centroids = np.array([(int(round(y)), int(round(x))) for y, x in centroids])
    # watershed
    masked = (labels < 2).astype("uint8")
    kernel2 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    masked = cv2.morphologyEx(masked, cv2.MORPH_CLOSE, kernel2)

    dist = ndi.distance_transform_edt(masked)
    markers = np.zeros(masked.shape, dtype=np.int32)
    for i, (y, x) in enumerate(centroids, start=1):
        markers[int(y), int(x)] = i
    labels = np.array(watershed(-dist, markers, mask=masked))
    labels, segmented_cells = filter_size(labels, len(centroids))
    # bounding boxes
    bounding_boxes = calculate_bb(segmented_cells)
    final_bb = []
    for bb in bounding_boxes:
        x1, y1, x2, y2 = bb
        if (x2 - x1) * (y2 - y1) < 16000 or (x2 - x1) < 25 or (y2 - y1) < 25:
            continue
        else:
            final_bb.append(bb)
    preds_dict[file.name] = final_bb
    preds_dict_pts[file.name] = centroids
    # plot_predictions(
    #     img_name=file.name,
    #     gt_points=ground_truth_pts,
    #     gt_boxes_raw=ground_truth,
    #     pred_points=centroids,
    #     pred_boxes=final_bb,
    #     dist_threshold=15,
    #     iou_threshold=0.3,
    #     it=idx,
    # )

bulk_eval(ground_truth, preds_dict, iou_threshold=0.4)
bulk_eval(
    ground_truth_pts,
    preds_dict_pts,
    iou_threshold=0.3,
    dist_threshold=15,
    points=True,
    plot=False,
)

INFO ilastik.app: config file location: <none>
INFO ilastik.app: Starting ilastik from /home/sdog/anaconda3/envs/microglia-env/bin
Starting ilastik from /home/sdog/anaconda3/envs/microglia-env/bin
INFO ilastik.app: Resetting lazyflow thread pool with 2 threads.
INFO ilastik.app: Configuring lazyflow RAM limit to 2.9GiB
INFO lazyflow.utility.memory: Available memory set to 2.9GiB


WARNING 2026-05-24 21:44:01,692 opConservationTracking 208173 132228525053440 Could not find any ILP solver
WARNING 2026-05-24 21:44:01,702 opStructuredTracking 208173 132228525053440 Could not find any ILP solver
WARNING 2026-05-24 21:44:01,704 structuredTrackingWorkflow 208173 132228525053440 Could not find any learning solver. Tracking will use flow-based solver (DPCT). Learning for tracking will be disabled!


INFO ilastik.shell.projectManager: Opening Project: ./ilastik_files/segmentation_final.ilp


0it [00:00, ?it/s]

INFO pyvips: VIPS: threadpool completed with 3 workers
INFO ilastik.applets.batchProcessing.batchProcessingApplet: Exporting to in-memory array.
INFO lazyflow.utility.bigRequestStreamer: Estimated RAM usage per pixel is 216.0B * safety factor (2.0)
INFO lazyflow.utility.bigRequestStreamer: determining blockshape assuming available_ram is 2.2GiB, split between 2 threads
INFO lazyflow.utility.bigRequestStreamer: Chose blockshape: (512, 512, 3)
INFO lazyflow.utility.bigRequestStreamer: Estimated RAM usage per block is 108.0MiB
DEBUG lazyflow.operators.classifierOperators: Features took 0.192964 seconds. Prediction took 1.6560709999999998 seconds. Subregion: start '[0, 0, 0]' stop '[512, 512, 3]'


...

35it [02:18,  3.89s/it]

INFO pyvips: VIPS: threadpool completed with 3 workers
INFO ilastik.applets.batchProcessing.batchProcessingApplet: Exporting to in-memory array.
INFO lazyflow.utility.bigRequestStreamer: Estimated RAM usage per pixel is 216.0B * safety factor (2.0)
INFO lazyflow.utility.bigRequestStreamer: determining blockshape assuming available_ram is 2.2GiB, split between 2 threads
INFO lazyflow.utility.bigRequestStreamer: Chose blockshape: (512, 512, 3)
INFO lazyflow.utility.bigRequestStreamer: Estimated RAM usage per block is 108.0MiB
DEBUG lazyflow.operators.classifierOperators: Features took 0.129435 seconds. Prediction took 1.947181 seconds. Subregion: start '[0, 0, 0]' stop '[512, 512, 3]'


36it [02:21,  3.93s/it]

Bounding box evaluation
Img:TPO_66_EV.tiff_6.png not found in GT but cells were localized
Img:TPO_63_TPO.tiff_10.png not found in GT but cells were localized
Evaluation output:
TP:31
FP:20
FN:15
Mean IOU:0.6677543291942565
F1-score: 0.6391752577319587
Pointwise evaluation
Img:TPO_67_TPO.tiff_9.png not found in GT but cells were localized
Img:TPO_66_EV.tiff_6.png not found in GT but cells were localized
Img:TPO_63_TPO.tiff_10.png not found in GT but cells were localized
Evaluation output:
TP:41
FP:47
FN:5
Mean Pixelwise Euc. Dist.:3.6085221541070616
F1-score: 0.6119402985074627


(41, 47, 5)